# 11 · Datos del SENAMHI: ¿sirven los pluviómetros reales?

> **No es un sistema de alerta ni una predicción para decidir evacuaciones.** Ante una emergencia llama al 115 (INDECI).

**Fuente:** Información recopilada y trabajada por el Servicio Nacional de Meteorología e Hidrología del Perú (SENAMHI), [Observatorio Nacional del Tiempo, Agua y Clima del Perú](https://observatorio.senamhi.gob.pe). Los términos ([senamhi.gob.pe](https://www.senamhi.gob.pe/?p=terminos-condiciones)) declaran la información «de uso público» pero reservan su publicación, reproducción, distribución y transformación a la institución, y el portal avisa que los datos no tienen control de calidad. **Por eso las series viven solo en `datos/senamhi/` (fuera de git y del espejo de Hugging Face) y aquí solo se muestran resultados agregados.** Falta la autorización por escrito; el borrador de la solicitud está en `docs/11-senamhi.md`.

Para correrlo: `uv run descargar-datos --solo senamhi` (opt-in; ~40 minutos porque pide mes a mes y espera 2 s entre llamadas).

**Teorías (escritas antes de mirar):**
- **T30** los pluviómetros de la cuenca media-alta (Autisha, Matucana, Santiago de Tuna...) detectan los días con huaico mejor que ERA5, porque ERA5 promedia 25 km y el huaico depende de tormentas locales;
- **T31** el pluviómetro del propio Chosica casi no mide la lluvia que activa las quebradas (cae arriba, no en el pueblo), pero que llueva algo en el pueblo ya separa los días con huaico;
- **T32** el nivel del río en Chosica concuerda solo en parte con el caudal de GloFAS.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

from illapa_research import estudio_eventos as ee, evaluacion_diaria as E, predictores as P, senamhi as S
from illapa_research.catalogo import cargar_todos

RAIZ = Path("..") if Path("../datos").exists() else Path(".")
CARPETA = RAIZ / "datos" / "senamhi"
HAY_DATOS = (CARPETA / "cache").exists() and any((CARPETA / "cache").glob("researchers__*"))
print("datos locales del SENAMHI:", "sí" if HAY_DATOS else "no (corre `descargar-datos --solo senamhi`)")
print(S.CITA)

## 1. Qué estaciones hay cerca de Chosica

Metadatos del catálogo público (estos sí se pueden publicar): `catalogo/senamhi_estaciones_cercanas.csv`, las estaciones meteorológicas e hidrológicas a menos de 45 km, con los meses de datos que publica el servicio.

In [ ]:
inv = pd.read_csv(RAIZ / "catalogo" / "senamhi_estaciones_cercanas.csv")
print(inv.groupby(["tipo", "condicion"]).size().unstack(fill_value=0))
inv[inv.meses_con_dato > 0].sort_values("km_a_chosica")[["nombre", "tipo", "categoria", "km_a_chosica", "altitud_m", "desde", "hasta", "meses_con_dato"]].head(25)

**Qué lectura da el servicio.** Hay dos puertas, ambas anónimas y las dos las usa la propia página «Datos hidrometeorológicos»: una serie **diaria histórica** por estación (precipitación y temperaturas extremas, de 1963 o 1989 hasta enero de 2014) y la **lectura mensual** de las estaciones vigentes (de 2013 o 2018 en adelante; las automáticas, por hora). Entre 2014 y 2017 no hay pluviómetro convencional publicado en Chosica; la estación hidrometeorológica automática de Chosica, con lluvia y nivel del río por hora, empieza en enero de 2001.

In [ ]:
if HAY_DATOS:
    c = S.Cliente(CARPETA)
    hist = S.red_historica(c)
    auto = S.red_automatica(c)
    cobertura = pd.concat({"convencionales": hist.notna().resample("YS").mean(), "automáticas": auto.notna().resample("YS").mean()}, axis=1)
    display((cobertura.loc["1989":] * 100).round(0).iloc[::2])

## 2. ¿Cuánto concuerda ERA5 con un pluviómetro real?

In [ ]:
if HAY_DATOS:
    ev = [pd.Timestamp(e.fecha) for e in cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True)]
    pred, _ = P.cargar_con_respaldo(RAIZ / "datos", rellenar_con_power=False)
    horario = ee.cargar_horario(RAIZ / "datos" / "open_meteo_horario_estudio.json")
    era_chosica = horario["chosica"]["precipitation"].resample("D").sum()
    filas = []
    for nombre, g, e_ in (
        ("Matucana (convencional) vs ERA5, 1989-2013", hist["matucana"]["1989":"2013"], pred["matucana"]["precipitation_sum"]),
        ("Matucana (automática) vs ERA5, 2013-2026", auto["matucana_auto"], pred["matucana"]["precipitation_sum"]),
        ("Chosica (automática) vs ERA5, 2001-2026", auto["chosica_auto"], era_chosica),
    ):
        a = S.acuerdo(g, e_, ev, 0)
        filas.append({"par": nombre, **{k: round(v, 2) for k, v in a.items()}})
    display(pd.DataFrame(filas).set_index("par"))

Lectura: ERA5 y el pluviómetro se parecen poco en el día a día (Spearman 0,4 a 0,6) y ERA5 da más lluvia en los extremos (p95 de 18 mm contra 7 mm en Matucana). **En Chosica, los días con huaico el pluviómetro del pueblo mide poco (mediana 0,9 mm) mientras ERA5 dice 5 mm. Aun así separa bien los días (AUC 0,80 la automática, 0,67 la convencional): T31 se cumple a medias.**

## 3. ¿Qué estación discrimina los días con huaico?

AUC de la lluvia del mismo día de cada estación (0,5 = nada), solo en temporada de lluvias.

In [ ]:
if HAY_DATOS:
    idx = pd.date_range("1984-01-01", "2026-10-29", freq="D")
    y = E.etiquetas(cargar_todos(solo_dia_exacto=True, sin_baja_confianza=True), idx)
    m = P.temporada_mask(idx)
    filas = []
    for nombre, R, a, b in (("convencionales 1990-2013", hist, 1990, 2013), ("automáticas 2014-2026", auto, 2014, 2026)):
        mm = m & (idx.year >= a) & (idx.year <= b)
        for col in R.columns:
            x = R[col].reindex(idx)[mm]
            ok = x.notna()
            if y[mm][ok].sum() >= 5 and ok.mean() > 0.3:
                filas.append({"red": nombre, "estacion": col, "auc": roc_auc_score(y[mm][ok], x[ok]), "dias_con_dato": int(ok.sum())})
    display(pd.DataFrame(filas).sort_values(["red", "auc"], ascending=[True, False]).round(3))

## 4. ¿Mejora el modelo diario del capítulo 05?

Se repite el modelo (regresión logística con regularización elegida dentro del entrenamiento, dejando una temporada fuera) con tres juegos de lluvia: ERA5, pluviómetros, y ambos. Los pluviómetros dan la lluvia **del día**, así que es un *nowcast* (el valor diario se lee a la mañana siguiente), no un pronóstico.

In [ ]:
if HAY_DATOS:
    r1 = S.experimento_pluviometros(RAIZ, hist, 1990, 2013)
    print(r1["n_dias"], "días,", r1["n_eventos"], "con huaico,", r1["n_temporadas"], "temporadas")
    display(r1["tabla"][["ap", "ap_base", "auc", "brier_skill", "aciertos_con_5pc_alarmas", "aciertos_con_10pc_alarmas"]].round(3))
    print("pluviómetros vs ERA5 (AP):", {k: round(v, 3) for k, v in S.diferencia_ap(r1["y"], r1["p"]["pluviómetros del día"], r1["p"]["ERA5 lluvia del día"], r1["ts"]).items()})

In [ ]:
if HAY_DATOS:
    red_auto = auto[[c for c in auto.columns if c not in ("nana_auto",)]]
    r2 = S.experimento_pluviometros(RAIZ, red_auto, 2014, 2026, separar_valle=False)
    print(r2["n_dias"], "días,", r2["n_eventos"], "con huaico,", r2["n_temporadas"], "temporadas; estaciones:", list(red_auto.columns))
    display(r2["tabla"][["ap", "ap_base", "auc", "brier_skill", "aciertos_con_5pc_alarmas", "aciertos_con_10pc_alarmas"]].round(3))
    for a, b in (("pluviómetros del día", "ERA5 lluvia del día"), ("ERA5 día + antecedentes + pluviómetros", "ERA5 día + antecedentes")):
        print(a, "vs", b, {k: round(v, 3) for k, v in S.diferencia_ap(r2["y"], r2["p"][a], r2["p"][b], r2["ts"]).items()})

## 5. Nivel del río en Chosica contra el caudal de GloFAS

In [ ]:
if HAY_DATOS:
    from scipy.stats import spearmanr

    ch = c.serie("47278214")
    nivel = pd.to_numeric(ch["N_NIVELAGUA"], errors="coerce")
    n_h = nivel.resample("D").count()
    nivel_d = nivel.resample("D").mean().where(n_h >= 20)
    glofas = np.log(P.cargar_caudal(RAIZ / "datos" / "open_meteo_caudal_glofas.json")["rimac-chosica"])
    j = pd.concat([nivel_d.rename("nivel_m"), glofas.rename("log_caudal_glofas")], axis=1).dropna().loc["2001":]
    print(len(j), "días; Spearman nivel (m) vs log caudal GloFAS:", round(spearmanr(j.nivel_m, j.log_caudal_glofas)[0], 2))

## Lo que no se puede concluir

- Son pocas estaciones y los registros convencionales terminan en enero de 2014; entre 2014 y 2017 la red de montaña casi no tiene datos públicos.
- No hay control de calidad ni historial de avisos descargable: el servicio solo publica los avisos vigentes (WMS sin dimensión de tiempo), así que no se puede comparar con los avisos oficiales pasados.
- Las series diarias convencionales se leen a las 07:00 y cubren las 24 horas previas; aquí se alinean con el día de calendario del mismo valor, que fue el desfase que más concordó con ERA5.
- La mejora se mide dentro de ventanas cortas (25 y 13 temporadas, 53 y 51 días con huaico): los intervalos son anchos aunque la diferencia es clara.